In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!apt-get install -y wget


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wget is already the newest version (1.21.2-2ubuntu1.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
!ls "/content/drive/MyDrive/NSL-KDD"


'index[1].html'     'KDDTest-21[1].arff'  'KDDTrain+_20Percent[1].arff'
'KDDTest1[1].jpg'   'KDDTest-21[1].txt'   'KDDTrain+_20Percent[1].txt'
'KDDTest+[1].arff'  'KDDTrain1[1].jpg'	  'KDDTrain+[2].txt'
'KDDTest+[1].txt'   'KDDTrain+[1].arff'


In [ ]:
train_path = "/content/drive/MyDrive/NSL-KDD/KDDTrain+[1].txt"
test_path = "/content/drive/MyDrive/NSL-KDD/KDDTest+[1].txt"


In [ ]:
train_path = "/content/drive/MyDrive/NSL-KDD/KDDTrain+[2].txt"
test_path = "/content/drive/MyDrive/NSL-KDD/KDDTest+[1].txt"



In [ ]:
import pandas as pd

column_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label"
]

train_df = pd.read_csv(train_path, names=column_names)
test_df = pd.read_csv(test_path, names=column_names)

train_df.head()


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,tcp,ftp_data,SF,491,0,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
0,udp,other,SF,146,0,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
0,tcp,private,S0,0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
0,tcp,http,SF,232,8153,0,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
0,tcp,http,SF,199,420,0,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [ ]:
print(train_df.head())
print(train_df.shape)
print(test_df.shape)


  duration protocol_type service  flag  src_bytes  dst_bytes  land  \
0      tcp      ftp_data      SF   491          0          0     0   
0      udp         other      SF   146          0          0     0   
0      tcp       private      S0     0          0          0     0   
0      tcp          http      SF   232       8153          0     0   
0      tcp          http      SF   199        420          0     0   

   wrong_fragment  urgent  hot  ...  dst_host_srv_count  \
0               0       0    0  ...                0.17   
0               0       0    0  ...                0.00   
0               0       0    0  ...                0.10   
0               0       0    0  ...                1.00   
0               0       0    0  ...                1.00   

   dst_host_same_srv_rate  dst_host_diff_srv_rate  \
0                    0.03                    0.17   
0                    0.60                    0.88   
0                    0.05                    0.00   
0           

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()

for col in ['protocol_type', 'service', 'flag']:
    combined = pd.concat([train_df[col], test_df[col]], axis=0)

    label_enc.fit(combined)

    train_df[col] = label_enc.transform(train_df[col])
    test_df[col] = label_enc.transform(test_df[col])


In [ ]:
train_df['label'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['label'] = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)


In [ ]:
X_train = train_df.drop(['label'], axis=1)
y_train = train_df['label']

X_test = test_df.drop(['label'], axis=1)
y_test = test_df['label']


In [ ]:
# STEP 6: Reduce to 4 features (works for DataFrames)
X_train_small = X_train.iloc[:, :4].to_numpy()
X_test_small  = X_test.iloc[:, :4].to_numpy()

input_shape = X_train_small.shape[1]  # = 4
print("X_train_small shape:", X_train_small.shape)
print("X_test_small shape:", X_test_small.shape)



X_train_small shape: (125973, 4)
X_test_small shape: (22544, 4)


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np

# 1️⃣ Encode categorical columns
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_test[col]]).astype(str)
        le.fit(combined)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col]  = le.transform(X_test[col].astype(str))

# 2️⃣ Convert everything to NumPy float32 arrays
X_train = X_train.to_numpy(dtype=np.float32)
X_test  = X_test.to_numpy(dtype=np.float32)

# 3️⃣ Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# 4️⃣ Reduce to 4 features
X_train_small = X_train[:, :4]
X_test_small  = X_test[:, :4]

print("✅ All features are numeric now")
print("X_train_small dtype:", X_train_small.dtype)
print("X_train_small shape:", X_train_small.shape)






✅ All features are numeric now
X_train_small dtype: float32
X_train_small shape: (125973, 4)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

input_shape = X_train_small.shape[1]

model = Sequential([
    Input(shape=(input_shape,)),
    Dense(12, activation='relu'),
    Dropout(0.4),
    Dense(6, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    X_train_small,
    y_train,
    epochs=8,          # fewer epochs to control accuracy
    batch_size=128,    # larger batch = weaker learning
    validation_split=0.2,
    verbose=1
)



Epoch 1/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9452 - loss: 0.2273 - val_accuracy: 1.0000 - val_loss: 2.1277e-04
Epoch 2/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 1.0000 - loss: 0.0113 - val_accuracy: 1.0000 - val_loss: 3.9080e-06
Epoch 3/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 0.0068 - val_accuracy: 1.0000 - val_loss: 1.8383e-07
Epoch 4/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 1.0000 - val_loss: 9.9262e-09
Epoch 5/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 0.0042 - val_accuracy: 1.0000 - val_loss: 7.0763e-10
Epoch 6/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 0.0037 - val_accuracy: 1.0000 - val_loss: 6.0109e-11
Epoch 7/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 5.4719e-12
Epoch 8/8
788/788 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 1.0000 - loss: 0.002

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate on REDUCED test features
loss, accuracy = model.evaluate(X_test_small, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")   # ✅ percentage

# Predict
y_pred = (model.predict(X_test_small) > 0.5).astype(int)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

